In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [3]:
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("data/pokeData.csv")
)

In [4]:
charmander = df.filter(F.col("pokemon_id") == 4)

In [5]:
charmander.show()

+----------+----------+--------+-------------------+----------+
|pokemon_id|      name|height_m|         updated_at|batch_date|
+----------+----------+--------+-------------------+----------+
|         4|Charmander|     0.6|2026-10-10 10:00:00|2026-10-10|
|         4|Charmander|     0.7|2026-10-12 08:30:00|2026-10-12|
|         4|Charmander|     0.8|2026-10-14 08:00:00|2026-10-14|
+----------+----------+--------+-------------------+----------+



In [6]:
window = Window.partitionBy("pokemon_id").orderBy(
    F.col("updated_at").desc()
)

In [7]:
scd1 = (
    charmander
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [8]:
scd1.select("pokemon_id", "name", "height_m", "updated_at").show()

+----------+----------+--------+-------------------+
|pokemon_id|      name|height_m|         updated_at|
+----------+----------+--------+-------------------+
|         4|Charmander|     0.8|2026-10-14 08:00:00|
+----------+----------+--------+-------------------+



In [9]:
history_window = Window.partitionBy("pokemon_id").orderBy("updated_at")

In [10]:
scd2 = (
    charmander
    .withColumn("valid_to", F.lead("updated_at").over(history_window))
    .withColumn(
        "is_current",
        F.col("valid_to").isNull()
    )
)

In [11]:
scd2.select(
    "pokemon_id", "name", "height_m",
    "updated_at", "valid_to", "is_current"
).orderBy("updated_at").show(truncate=False)

+----------+----------+--------+-------------------+-------------------+----------+
|pokemon_id|name      |height_m|updated_at         |valid_to           |is_current|
+----------+----------+--------+-------------------+-------------------+----------+
|4         |Charmander|0.6     |2026-10-10 10:00:00|2026-10-12 08:30:00|false     |
|4         |Charmander|0.7     |2026-10-12 08:30:00|2026-10-14 08:00:00|false     |
|4         |Charmander|0.8     |2026-10-14 08:00:00|NULL               |true      |
+----------+----------+--------+-------------------+-------------------+----------+



In [12]:
spark.stop()